In [12]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import os
import json
import torch 
import torch.nn as nn

pd.set_option("display.max_colwidth", None)



In [ ]:
#Get the videos json
videos = os.path.join("..", "Videos.json")

#Open the json
with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

#Turn the json into a pandas dataframe with proper headings     
df = pd.json_normalize(data["videos"])

print(df.columns)
print(df.head())


Index(['id', 'Sign', 'filepath', 'HamNoSys', 'concept_url', 'HandLandmark'], dtype='str')
   id      Sign                                           filepath  \
0   1     April  C:\Users\mccor\Documents\Y4\Dis\Project\Data\E...   
1   2    Athens  C:\Users\mccor\Documents\Y4\Dis\Project\Data\E...   
2   3    August  C:\Users\mccor\Documents\Y4\Dis\Project\Data\E...   
3   4    Berlin  C:\Users\mccor\Documents\Y4\Dis\Project\Data\E...   
4   5  February  C:\Users\mccor\Documents\Y4\Dis\Project\Data\E...   

                                          HamNoSys  \
0     
1                 
2         
3                                         
4                             

                                         concept_url  \
0  https://www.sign-lang.uni-hamburg.de/dicta-sig...   
1  https://www.sign-lang.uni-hamburg.de/dicta-sig...

In [ ]:
#find the path to each landmark json in the videos json
Hand_Landmarks_paths = df["HandLandmark"]

Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]


In [ ]:

#Empty list for storing each dataframe of the handlandmarks
Hand_landmarks_dfs = []

for path in Hand_Landmarks:
    
    with open(path, "r") as f:
        data = json.load(f)
        
    #Create dataframes
    temp_df = pd.json_normalize(data)
    temp_df["source"] = path
    
    Hand_landmarks_dfs.append(temp_df)

In [ ]:
#Take each handlandmark df and make it into one big df
Hand_Landmark_df = pd.concat(Hand_landmarks_dfs, ignore_index=True)

In [23]:
#Hand_Landmark_df.head(1)



In [ ]:
rows = []

#For each video
for _, video_row in Hand_Landmark_df.iterrows():
    video_path = video_row["video_path"]#Get each video path
    fps = video_row["fps"]#Get the fps for those videos
    source = video_row.get("source",None)#
    
    #for all the frames in current video
    for frame in video_row["frames"]:
        frame_index = frame["frame_index"] #Set frame index
        time_sec = frame["time_sec"]#Set the time of frame
        
        #Then for each hand in current frame
        for hand in frame["hands"]:
            handedness = hand["handedness"]#which hand
            handedness_score = hand["handedness_score"]#How sure that this is the right hand
            hand_index = hand["hand_index"]#The index of hand
            
            #For each landmark in hand
            for lm in hand["landmarks"]:
                rows.append({
                    "video_path": video_path,
                    "fps": fps,
                    "source": source,
                    "frame_index": frame_index,
                    "time_sec": time_sec,
                    "hand_index": hand_index,
                    "handedness": handedness,
                    "handedness_score": handedness_score,
                    "landmark_id": lm["id"],
                    "landmark_name": lm["name"],
                    "x": lm["x"],
                    "y": lm["y"],
                    "z": lm["z"],
                })
                
Hand_landmarks_df = pd.DataFrame(rows)

In [19]:
print(Hand_landmarks_df.head())

                                                        video_path   fps  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4  25.0   

                                                                  source  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   

   frame_index  time_sec  hand_index handedness  handedness_score  \
0            0   

In [ ]:
#Need to piivot table so that it is done by frame instead of landmark

wide_df = Hand_landmarks_df.pivot_table(
    index=["video_path", "frame_index", "time_sec", "source"],
    columns=["handedness", "landmark_name"],
    values=["x", "y", "z"]
)

#Create readable headings
wide_df.columns = [
    f"{hand}_{landmark}_{axis}"
    for axis, hand, landmark in wide_df.columns
]

wide_df = wide_df.reset_index()
wide_df = wide_df.fillna(0.0)

In [22]:
print(wide_df.head())
print(wide_df.shape)

                                                        video_path  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   

   frame_index  time_sec  \
0            0      0.00   
1            1      0.04   
2            5      0.20   
3            6      0.24   
4            7      0.28   

                                                                  source  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
4  C:\Users\mccor\Documents\Y4